# Initialization

In [0]:
%run ../../utils/config

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.erp_px_cat_g1v2")

In [0]:
df.limit(2).display()

# Silver Transformations

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df.limit(2).display()

## Normalize Maintenance Flag to Boolean

In [0]:
df = df.withColumn(
      "MAINTENANCE",
      F.when(F.upper(col("MAINTENANCE")) == "YES", F.lit(True))
       .when(F.upper(col("MAINTENANCE")) == "NO", F.lit(False))
       .otherwise(None)
)

In [0]:
df.limit(2).display()

## Renaming Columns

In [0]:
RENAME_MAP = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity checks of dataframe

In [0]:
df.limit(2).display()

# Writing Silver Table

In [0]:
from delta.tables import DeltaTable 

TARGET_TABLE = TABLES["erp_cat"]
PK_COL = "category_id"

if not spark.catalog.tableExists(TARGET_TABLE):
    df.write.format("delta").saveAsTable(TARGET_TABLE)
else:
    delta_target = DeltaTable.forName(spark, TARGET_TABLE)
    (
        delta_target.alias("target")
        .merge(
            df.alias("source"),
            f"target.{PK_COL} = source.{PK_COL}"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# Collect MERGE Metrics

In [0]:
dt = DeltaTable.forName(spark, TARGET_TABLE)
metrics = dt.history(1).select("operationMetrics").collect()[0]
op_metrics = metrics["operationMetrics"]

# Sanity checks of silver table

In [0]:
spark.sql(f"SELECT * FROM {TARGET_TABLE} LIMIT 10").display()

# Data Quality Checks

In [0]:
df_check = spark.table(TARGET_TABLE)
total_rows   = df_check.count()
null_pk      = df_check.filter(F.col(PK_COL).isNull()).count()
duplicate_pk = total_rows - df_check.select(PK_COL).distinct().count()

print(f"[QC] {TARGET_TABLE}")
print(f"  Total rows    : {total_rows}")
print(f"  Null PKs      : {null_pk}")
print(f"  Duplicate PKs : {duplicate_pk}")

try:
    assert total_rows    > 0,  f"[QC FAILED] {TARGET_TABLE} is empty"
    assert null_pk      == 0,  f"[QC FAILED] {null_pk} null PKs in {PK_COL}"
    assert duplicate_pk == 0,  f"[QC FAILED] {duplicate_pk} duplicate PKs in {PK_COL}"
    qc_status = "PASS"
    qc_message = "All checks passed"
    print("[QC PASSED]")
except AssertionError as e:
    qc_status = "FAIL"
    qc_message = str(e)
    print(f"[QC FAILED] {qc_message}")

## Write Audit Log

In [0]:
from datetime import datetime

row = [{
    "notebook_name": "silver_erp_px_cat_g1v2",
    "target_table": TARGET_TABLE,
    "run_timestamp": datetime.now(),
    "rows_inserted": int(op_metrics.get("numTargetRowsInserted", 0)),
    "rows_updated": int(op_metrics.get("numTargetRowsUpdated", 0)),
    "rows_deleted": int(op_metrics.get("numTargetRowsDeleted", 0)),
    "qc_status": qc_status,
    "qc_message": qc_message
}]

df_audit = spark.createDataFrame(row)
df_audit.write.mode("append").format("delta").saveAsTable(TABLES["audit_log"])

##Sanity Check - Audit Log

In [0]:
spark.sql(f"SELECT * FROM {TABLES["audit_log"]} ORDER BY run_timestamp DESC").display()